# Embeddings das cartas (Etapa 2)

*(Feito com ClaudeAI: https://claude.ai/share/b3536ad5-de2f-4b3f-a788-ba1c9a17f7da)*

Gera o embedding de cada carta a partir de `embedding_text_structured`
(`card_documents.jsonl`), usando um modelo local (sem API, sem custo).
Essa e' so a etapa 3.1 do plano (gerar e salvar os vetores) -- a funcao de
busca de verdade, com indice e interface de consulta, e' a etapa 3.2,
separada.

## Configuracao

In [ ]:
import json
import numpy as np

CARD_DOCUMENTS_PATH = "Dados/card_documents.jsonl"
MODEL_NAME = "BAAI/bge-small-en-v1.5"
TEXT_FIELD = "embedding_text_structured"  # qual das 3 representacoes usar

EMBEDDINGS_PATH = "Dados/embeddings.npy"        # os vetores, um por carta
EMBEDDINGS_META_PATH = "Dados/embeddings_meta.json"  # id/nome de cada carta, na mesma ordem

## Carregar os documentos de carta

So carregamos o campo de texto escolhido e um identificador (`id` do
Scryfall + `name`, pra conseguir religar cada vetor a carta depois).
`embedding_text_structured` pode vir vazio pra pouquissimos casos raros
(carta sem type_line, por exemplo) -- filtramos esses fora em vez de gerar
um embedding de string vazia, que nao teria significado nenhum.

In [2]:
names, texts = [], []
with open(CARD_DOCUMENTS_PATH, encoding="utf-8") as f:
    for line in f:
        card = json.loads(line)
        text = card.get(TEXT_FIELD)
        if not text:
            continue
        names.append(card["name"])
        texts.append(text)

print(f"{len(texts)} cartas carregadas (de {sum(1 for _ in open(CARD_DOCUMENTS_PATH, encoding='utf-8'))} no arquivo)")

34599 cartas carregadas (de 34599 no arquivo)


## Gerar os embeddings

Modelos BGE tem uma convencao de uso assimetrica: **documentos** sao
embedados como estao, mas **consultas** de busca precisam de um prefixo
de instrucao (`"Represent this sentence for searching relevant passages:
"`) pra funcionar bem -- isso vem de como o modelo foi treinado.
Documentamos isso aqui porque a funcao de busca (etapa 3.2) vai precisar
aplicar esse prefixo na hora de embedar a consulta do usuario, e' facil
esquecer e prejudicar a qualidade da busca sem nenhum erro aparente.

O encode roda em lote (`batch_size`) pra nao estourar memoria com 38 mil
textos de uma vez.

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)
print(f"Modelo carregado: {MODEL_NAME} (max_seq_length={model.max_seq_length})")

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,  # normalizado -> similaridade de cosseno vira so um produto escalar
)
print(f"Embeddings gerados: shape {embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo carregado: BAAI/bge-small-en-v1.5 (max_seq_length=512)


Batches:   0%|          | 0/541 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Salvar os vetores e o mapeamento

In [ ]:
np.save(EMBEDDINGS_PATH, embeddings)

with open(EMBEDDINGS_META_PATH, "w", encoding="utf-8") as f:
    json.dump({"names": names}, f, ensure_ascii=False)

print(f"Embeddings salvos em {EMBEDDINGS_PATH} ({embeddings.nbytes / 1e6:.1f} MB)")
print(f"Metadados salvos em {EMBEDDINGS_META_PATH}")

## Sanity check (nao e' a busca final -- isso e' a etapa 3.2)

So pra confirmar que os vetores gerados fazem sentido antes de fechar essa
etapa: uma consulta simples, busca por cosseno "na mao" (sem indice, sem
otimizacao -- 38 mil vetores cabem em memoria numa conta direta) e os
top-5 resultados. A funcao de busca de verdade, com interface e tratamento
de erro, e' a proxima etapa.

In [ ]:
query = "creature that destroys artifacts"
query_vec = model.encode(
    ["Represent this sentence for searching relevant passages: " + query],
    normalize_embeddings=True,
)[0]

scores = embeddings @ query_vec
top_k = np.argsort(-scores)[:5]

print(f"Consulta: {query!r}\n")
for i in top_k:
    print(f"  {scores[i]:.3f}  {names[i]}")